📦 Bloque 0 — Imports estándar

In [ ]:
import pandas as pd
import numpy as np
import sys
import importlib
import json
import os
from os import path
from pathlib import Path

🧩 Bloque 1 — Import módulo `functions` + carga config JSON

In [ ]:
# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/banigualdad/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path(os.environ["PROJECT_ROOT"]).resolve()
FUNCTIONS_DIR = PROJECT_ROOT / "functions"
CONFIG_DIR = PROJECT_ROOT / "config" / "CCLA"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:", CONFIG_DIR, "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"

try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)

    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))

except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"sys.path{sys.path[0] if sys.path else '<vacío>'}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "config_volver_flujo_2.json")

try:
    with open(config_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()) if isinstance(data, dict) else type(data))

except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")

⚙️ Bloque 2 — Conexión SQL Server

In [ ]:
# --- Parámetros iguales a los tuyos ---
# --- Parámetros desde config ---
server = data["server_config"]["server"]
database = data["server_config"]["database"]
schema = data["server_config"]["schema"]

driver = "ODBC Driver 17 for SQL Server"

# 1) ODBC connection string (pyodbc)
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# 2) SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise",
)

📂 Bloque 3 — Rutas y hojas preferidas

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "CCLA" / "VOLVEK FLUJO 2").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

EXTS = (".xlsx",)

PREFERRED_SHEETS = ["Base Flujo 2.0", "Base cesantia SURA Flujo 2.0", "Base Flujo 2"]
FALLBACK_SHEET_INDEX = 0

🧹 Bloque 4 — Detección automática de archivo y hoja + preview

In [ ]:
info = fun.get_latest_excel_and_sheet(
    RUTA_ARCHIVOS,
    EXTS,
    PREFERRED_SHEETS,
    fallback_sheet_index=FALLBACK_SHEET_INDEX,
    recursive=False,
    engine="openpyxl",
)

archivo_origen   = info["archivo_origen"]
excel_path_used  = info["excel_path_used"]
_tmp_copy_path   = info["tmp_copy_path"]
target           = info["target_sheet"]
sheet_names      = info["sheet_names"]

# (Opcional) preview sin header
df_opexcel = fun.read_excel_safe_no_header(excel_path_used, target)
fun.pretty_table(df_opexcel, n=5)

📑 Bloque 5 — Lectura + normalización de headers

In [ ]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = fun.read_excel_safe(io, target)

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "none": "", "NaN": "", "nan": ""})
df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]
print("Encabezados normalizados:", list(df_raw.columns))
fun.pretty_table(df_raw, n=5)

📄 Archivo más reciente: Base cesantia SURA Flujo 2.0 NOVIEMBRE 2025.xlsx  |  2025-12-29 16:23:10
Hojas: ['Hoja1', 'Base Flujo 2.0']
✅ Hoja objetivo: Base Flujo 2.0


🧾 Bloque 6 — Mapeo de columnas (Origen → Destino)

In [ ]:
# Origen → Destino
foliocredito        = fun.pick(df_raw, "n_operacion", "no_operacion", "noperacion", "operacion", "folio")
NombreAfiliado      = fun.pick(df_raw, "afinom", "nombre_afiliado", "nombre")
rutafiliado         = fun.pick(df_raw, "afirut", "rut_afiliado", "rut")
dvafiliado          = fun.pick(df_raw, "afirutdv", "dv_afiliado", "dv")
Plazo               = fun.pick(df_raw, "crecuotot", "plazo")
MontoBruto          = fun.pick(df_raw, "cresolmon", "monto_bruto", "monto")
fechaPrimerVto      = fun.pick(df_raw, "fecinicob", "fecha_primer_vto", "fec_ini_cob")
FechaUltimoVto      = fun.pick(df_raw, "fectercob", "fecha_ultimo_vto", "fec_ter_cob")
ValorCuota          = fun.pick(df_raw, "crecuomon", "valor_cuota")
PrimaBruta_src      = fun.pick(df_raw, "prima_cliente", "prima_bruta", "prima_cliente_")
PrimaNeta_src       = fun.pick(df_raw, "prima_exenta", "prima_neta")
Com25_src           = fun.pick(df_raw, "comision_caja_25", "comision_caja_25_", "comision_25")
Com26_src           = fun.pick(df_raw, "comision_caja_variable_26", "comision_variable_26", "comision_26")
Producto            = fun.pick(df_raw, "producto")
FolioOrigen         = fun.pick(df_raw, "folio_original", "folio_origen")
FechaOrigen         = fun.pick(df_raw, "fecha_otorgamiento_folio_original", "fecha_origen")

df_can = pd.DataFrame({
    "foliocredito": foliocredito,
    "NombreAfiliado": NombreAfiliado,
    "rutafiliado": rutafiliado,
    "dvafiliado": dvafiliado,
    "Plazo": Plazo,
    "MontoBruto": MontoBruto,
    "fechaPrimerVto": fechaPrimerVto,
    "FechaUltimoVto": FechaUltimoVto,
    "ValorCuota": ValorCuota,
    "PrimaBruta": PrimaBruta_src,
    "PrimaNeta": PrimaNeta_src,
    "Comision25": Com25_src,
    "ComisionVariable26": Com26_src,
    "Producto": Producto,
    "FolioOrigen": FolioOrigen,
    "FechaOrigen": FechaOrigen,
    "Nombre_de_archivo": archivo_origen.name,
})

fun.pretty_table(df_can, n=5)

Encabezados normalizados: ['noperacion', 'afinom', 'afirut', 'afirutdv', 'crecuotot', 'cresolmon', 'fecinicob', 'fectercob', 'crecuomon', 'prima_cliente', 'prima_exenta', 'comision_caja_25', 'comision_caja_variable_26', 'producto', 'folio_original', 'fecha_otorgamiento_folio_original']


,noperacion,afinom,afirut,afirutdv,crecuotot,cresolmon,fecinicob,fectercob,crecuomon,prima_cliente,prima_exenta,comision_caja_25,comision_caja_variable_26,producto,folio_original,fecha_otorgamiento_folio_original
0,001027701493,HUGO LAVADO ZUNIGA,12900600,5,24,1910646,20240731,20260731,69908,3821,3210.924369747899,802.7310924369748,834.8403361344538,REPROGRAMACION,137000180283,20181204
1,135000440582,JOSE MORALES OLAVARRIA,12131369,3,59,3995085,20250331,20300228,97867,7990,6714.285714285715,1678.5714285714287,1745.7142857142858,REPROGRAMACION,135000409904,20190109
2,133000424834,ELIECER HERNANDEZ LOPEZ,14186783,0,61,1212879,20221028,20271130,43024,2426,2038.655462184874,509.6638655462185,530.0504201680673,REPROGRAMACION,003000807733,20181108
3,023000466732,VIVIANA COLLAO FLORES,6804061,2,70,2792237,20220628,20280430,98236,5584,4692.436974789916,1173.109243697479,1220.0336134453783,REPROGRAMACION,023000461707,20190712
4,172000093607,CRISTIAN VALENZUELA CARMONA,12246591,8,57,4363338,20230825,20280531,108301,8727,7333.6134453781515,1833.4033613445379,1906.7394957983195,REPROGRAMACION,172000087202,20190927


🗂️ Bloque 7 — Canonicalización nombre de archivo

In [ ]:
nombre_canonico = fun.canonicalizar_nombre_archivo(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)

df_can["Nombre_de_archivo"] = nombre_canonico

,foliocredito,NombreAfiliado,rutafiliado,dvafiliado,Plazo,MontoBruto,fechaPrimerVto,FechaUltimoVto,ValorCuota,PrimaBruta,PrimaNeta,Comision25,ComisionVariable26,Producto,FolioOrigen,FechaOrigen,Nombre_de_archivo
0,001027701493,HUGO LAVADO ZUNIGA,12900600,5,24,1910646,20240731,20260731,69908,3821,3210.924369747899,802.7310924369748,834.8403361344538,REPROGRAMACION,137000180283,20181204,Base cesantia SURA Flujo 2.0 NOVIEMBRE 2025.xlsx
1,135000440582,JOSE MORALES OLAVARRIA,12131369,3,59,3995085,20250331,20300228,97867,7990,6714.285714285715,1678.5714285714287,1745.7142857142858,REPROGRAMACION,135000409904,20190109,Base cesantia SURA Flujo 2.0 NOVIEMBRE 2025.xlsx
2,133000424834,ELIECER HERNANDEZ LOPEZ,14186783,0,61,1212879,20221028,20271130,43024,2426,2038.655462184874,509.6638655462185,530.0504201680673,REPROGRAMACION,003000807733,20181108,Base cesantia SURA Flujo 2.0 NOVIEMBRE 2025.xlsx
3,023000466732,VIVIANA COLLAO FLORES,6804061,2,70,2792237,20220628,20280430,98236,5584,4692.436974789916,1173.109243697479,1220.0336134453783,REPROGRAMACION,023000461707,20190712,Base cesantia SURA Flujo 2.0 NOVIEMBRE 2025.xlsx
4,172000093607,CRISTIAN VALENZUELA CARMONA,12246591,8,57,4363338,20230825,20280531,108301,8727,7333.6134453781515,1833.4033613445379,1906.7394957983195,REPROGRAMACION,172000087202,20190927,Base cesantia SURA Flujo 2.0 NOVIEMBRE 2025.xlsx


🏷️ Bloque 8 — Eliminación de filas finales nulas (trailing totals)

In [ ]:
df_can = fun.drop_trailing_mostly_null(
    df_can,
    null_check_exclude=("Nombre_de_archivo",),
    also_exclude_money_cols=("PrimaBruta", "PrimaNeta", "Comision25", "ComisionVariable26"),
    null_ratio_threshold=0.80,
    verbose=True,
)

Nombre original : Base cesantia SURA Flujo 2.0 NOVIEMBRE 2025.xlsx
Nombre canónico : Base cesantia SURA Flujo 2.0 202511.xlsx


🧮 Bloque 9 — Casting de tipos + preparación df_sql

In [ ]:
# --- Casting numérico ---
fun.cast_numeric_columns(
    df_can,
    bigint_cols=["foliocredito", "FolioOrigen"],
    int_cols=["rutafiliado", "Plazo", "fechaPrimerVto", "FechaUltimoVto", "FechaOrigen"],
    float_cols=["MontoBruto", "ValorCuota", "PrimaBruta", "PrimaNeta", "Comision25", "ComisionVariable26"],
)

# --- DV: char(1) mayúscula ---
fun.normalize_dv_column(df_can, "dvafiliado")

# --- Strings: trim y largo máximo ---
fun.trim_string_columns(df_can, {
    "NombreAfiliado": 510,
    "Producto": 510,
    "Nombre_de_archivo": 50,
})

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)

# --- Nulos en columnas críticas ---
fun.report_nulls(df_can, ["foliocredito", "rutafiliado", "dvafiliado", "FolioOrigen", "Nombre_de_archivo"])

# --- Construir df_sql ---
cols_sql = [
    "foliocredito", "NombreAfiliado", "rutafiliado", "dvafiliado", "Plazo", "MontoBruto",
    "fechaPrimerVto", "FechaUltimoVto", "ValorCuota", "PrimaBruta", "PrimaNeta",
    "Comision25", "ComisionVariable26", "Producto", "FolioOrigen", "FechaOrigen", "Nombre_de_archivo",
]

df_sql = fun.build_sql_frame(df_can, cols_sql)

print("\n📊 df_sql listo con columnas:", list(df_sql.columns))
fun.pretty_table(df_sql, n=5)

🧹 Última fila TOTAL detectada (sumas OK y muchos nulos). Eliminando…

🧾 Fila eliminada:


,foliocredito,NombreAfiliado,rutafiliado,dvafiliado,Plazo,MontoBruto,fechaPrimerVto,FechaUltimoVto,ValorCuota,PrimaBruta,PrimaNeta,Comision25,ComisionVariable26,Producto,FolioOrigen,FechaOrigen,Nombre_de_archivo
214,,,,,,,,,,1335126,1121954.621848739,280488.65546218475,291708.20168067224,,,,Base cesantia SURA Flujo 2.0 202511.xlsx


🔢 Bloque 10 — Extracción de valores de partición

In [ ]:
assert "Nombre_de_archivo" in df_sql.columns, "Falta la columna 'Nombre_de_archivo' en df_sql."

df_sql["Nombre_de_archivo"] = (
    df_sql["Nombre_de_archivo"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["Nombre_de_archivo"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚿 No se encontraron valores válidos en 'Nombre_de_archivo'.")

counts = (
    df_sql["Nombre_de_archivo"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} Nombre_de_archivo distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

✅ dtypes DESPUÉS:
 foliocredito                   Int64
NombreAfiliado        string[python]
rutafiliado                    Int64
dvafiliado                    object
Plazo                          Int64
MontoBruto                   float64
fechaPrimerVto                 Int64
FechaUltimoVto                 Int64
ValorCuota                   float64
PrimaBruta                   float64
PrimaNeta                    float64
Comision25                   float64
ComisionVariable26           float64
Producto              string[python]
FolioOrigen                    Int64
FechaOrigen                    Int64
Nombre_de_archivo     string[python]
dtype: object

🔎 Nulos en columnas críticas:
 - foliocredito: 0 nulos
 - rutafiliado: 0 nulos
 - dvafiliado: 0 nulos
 - FolioOrigen: 0 nulos
 - Nombre_de_archivo: 0 nulos

📦 df_sql listo con columnas: ['foliocredito', 'NombreAfiliado', 'rutafiliado', 'dvafiliado', 'Plazo', 'MontoBruto', 'fechaPrimerVto', 'FechaUltimoVto', 'ValorCuota', 'PrimaBruta', 

🧾 Bloque 11 — Carga a SQL Server (replace_partition / append)

In [ ]:
resumen = []

table_name = data["tablas_remotas"]["volvek_acumulado_flujo2"]

for nombre_archivo in vals:

    df_sub = df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]

    if df_sub.empty:
        print(f"⚠️ No se encontraron filas para Nombre_de_archivo = '{nombre_archivo}'. Se omite.")
        continue

    sql_count = f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table_name}
        WHERE Nombre_de_archivo = '{nombre_archivo}'
    """

    df_cnt = fun.query_to_df(
        sql=sql_count,
        connection_string=connection_string,
        engine="pandas",
        return_iter=False,
    )

    existing_count = int(df_cnt.iloc[0, 0]) if not df_cnt.empty else 0

    if existing_count > 0:
        print(
            f"♻️ Se encontró data previa para Nombre_de_archivo = "
            f"'{nombre_archivo}' ({existing_count} filas en {table_name})."
        )
        print("🧹 Eliminando filas anteriores para reemplazarlas...")

        summary = fun.df_to_db(
            df=df_sub,
            connection_string=connection_string,
            schema=schema,
            table=table_name,
            mode="replace_partition",
            partition_column="Nombre_de_archivo",
            partition_values=[nombre_archivo],
            engine="pandas",
        )

        accion = "reemplazo"
        filas_borradas = summary["rows_deleted"]
        print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {filas_borradas}")

    else:
        print(
            f"🆕 No hay data previa para Nombre_de_archivo = "
            f"'{nombre_archivo}'. Se insertará como archivo nuevo."
        )

        summary = fun.df_to_db(
            df=df_sub,
            connection_string=connection_string,
            schema=schema,
            table=table_name,
            mode="append",
            engine="pandas",
        )

        accion = "inserción nueva"

    filas_sub = len(df_sub)
    print(f"📥 Insertadas {filas_sub} filas para Nombre_de_archivo = '{nombre_archivo}'.")
    resumen.append((nombre_archivo, filas_sub, existing_count, accion))

print("\n📈 Resumen de carga por Nombre_de_archivo:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")


📄 Se detectaron 1 Nombre_de_archivo distintos en el df_sql:
   - Base cesantia SURA Flujo 2.0 202511.xlsx: 214 filas


🔄 Bloque 12 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [12]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE Nombre_de_archivo = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE Nombre_de_archivo = :nombre
    """)

    for nombre_archivo in vals:
        df_sub = df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para Nombre_de_archivo = '{nombre_archivo}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para Nombre_de_archivo = '{nombre_archivo}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para Nombre_de_archivo = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=table,
            con=conn,       
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para Nombre_de_archivo = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por Nombre_de_archivo:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")


♻️ Se encontró data previa para Nombre_de_archivo = 'Base cesantia SURA Flujo 2.0 202511.xlsx' (214 filas en dbo.VOLVEK_ACUMULADO_FLUJO2).
🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...
✅ Filas eliminadas en destino para 'Base cesantia SURA Flujo 2.0 202511.xlsx': 214
📥 Insertadas 214 filas para Nombre_de_archivo = 'Base cesantia SURA Flujo 2.0 202511.xlsx'.

📊 Resumen de carga por Nombre_de_archivo:
   - Base cesantia SURA Flujo 2.0 202511.xlsx: 214 filas cargadas (reemplazando 214 previas).


🗑️ Bloque 12 — Eliminación de archivo origen

In [ ]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\VOLVEK FLUJO 2\Base cesantia SURA Flujo 2.0 OCTUBRE 2025.xlsx


# SQL

### Bloque 13 — Query: archivos cargados

In [ ]:
query_1 = f"""
-- [VOLVEK_FLUJO2] Archivos cargados (desc)
SELECT DISTINCT Nombre_de_archivo
FROM {data["tablas_remotas"]["volvek_acumulado_flujo2"]}
ORDER BY Nombre_de_archivo DESC;
"""

df_q1 = fun.query_to_df(
    sql=query_1,
    connection_string=connection_string,
    engine="pandas",
)

fun.pretty_table(df_q1)

## CONSTRUCCIÓN DE BASES FINALES POR CORREDOR / FUENTE

### Bloque 14 — Query: DROP tabla final

In [ ]:
query_2 = f"""
DROP TABLE IF EXISTS {data["tablas_remotas"]["flujo2_final_acumulado"]};
"""

fun.exec_sql(query_2, connection_string)

In [ ]:
query_3 = f"""
SELECT *,
       5698774 AS POLIZA,
       SUBSTRING(Nombre_de_archivo,
                 LEN(Nombre_de_archivo) - CHARINDEX('.', REVERSE(Nombre_de_archivo)) - 5,
                 6) AS MES_Recaudacion,
       6270 AS PLAN_TECNICO,
       4 AS PLAZO_CUOTAS,
       'Credito Consumo' AS Negocio
INTO {data["tablas_remotas"]["flujo2_final_acumulado"]}
FROM {data["tablas_remotas"]["volvek_acumulado_flujo2"]};
"""

fun.exec_sql(query_3, connection_string)